In [112]:
import requests
import pandas as pd
import altair as alt

In [19]:
# stablecoin historical prices
url = "https://stablecoins.llama.fi/stablecoinprices"
response = requests.get(url)
data = response.json()
df_list = []
for d in data:
    date = d["date"]
    coins = pd.DataFrame(list(d["prices"].items()), columns = ["stablecoin", "price"])
    # print(d["prices"])
    coins["date"] = date
    df_list.append(coins)
df = pd.concat(df_list)
df["date"] = pd.to_datetime(df["date"], unit="s")
df = df[df["date"] != "1970-01-01"]

In [108]:
def get_stablecoin_circulating_data():
    url = "https://stablecoins.llama.fi/stablecoins"
    response = requests.get(url)
    data = response.json()["peggedAssets"]
    dict_list = []
    for coin in data:
        # for come reason this is very confusing to work
        circ = list(coin["circulating"].values())[0]
        d = {"id": coin["id"], "name": coin["name"], "symbol": coin["symbol"], "circulating": circ}
        dict_list.append(d)
    df = pd.DataFrame(dict_list)
    return df

def get_top_stablecoins(df):
    df_sorted = df.sort_values("circulating", ascending = False)
    return df_sorted.head(5)["symbol"].values
df = get_stablecoin_circulating_data()
topstables = get_top_stablecoins(df)

In [111]:
df

,id,name,symbol,circulating
0,1,Tether,USDT,1.834648e+11
1,2,USD Coin,USDC,7.397355e+10
2,209,Sky Dollar,USDS,6.668091e+09
3,5,Dai,DAI,4.792426e+09
4,262,World Liberty Financial USD,USD1,4.187850e+09
...,...,...,...,...
417,427,JPYSC,JPYSC,1.253341e+08
418,428,KRWQ,KRWQ,5.456213e+05
419,429,KRW1,KRW1,8.395460e+03
420,436,Revolut Euro,EURR,2.906843e+05


In [113]:
def plot_stablecoins_market_dominance(df):
    # 1. Sort dataframe by circulating supply descending
    df_sorted = df.sort_values("circulating", ascending=False)
    
    # 2. Extract top 5 and group the rest into "Others"
    top_5 = df_sorted.head(5).copy()
    others_sum = df_sorted.iloc[5:]["circulating"].sum()
    
    # Create a combined dataframe for plotting
    others_df = pd.DataFrame([{"symbol": "Others", "circulating": others_sum}])
    plot_df = pd.concat([top_5[["symbol", "circulating"]], others_df], ignore_index=True)
    
    # 3. Build the pie chart in Altair
    chart = (
        alt.Chart(plot_df)
        .mark_arc(outerRadius=120)
        .encode(
            theta=alt.Theta(field="circulating", type="quantitative"),
            color=alt.Color(
                field="symbol",
                type="nominal",
                sort=None,
                title="Stablecoin",
            ),
            tooltip=["symbol", alt.Tooltip("circulating:Q", format=",.0f")],
        )
        .properties(
            title="Top 5 Stablecoins by Circulating Supply",
            width=400,
            height=400,
        )
    )

    return chart

chart.show() # or chart in Jupyter

alt.Chart(...)